In [4]:
import re
import httpx
import os
from dotenv import load_dotenv
import anthropic

In [5]:
load_dotenv()
api_key = os.getenv('API_KEY')
client = anthropic.Anthropic(api_key=api_key)

In [39]:
class Agent:
    def __init__(self,system=''):
        self.system = system
        self.messages = []
    
    def __call__(self,message):
        self.messages.append({'role':'user','content':message})
        result = self.execute()
        self.messages.append({'role':'assistant','content':result})
        return result
    
    def execute(self):
        response = client.messages.create(
            model='claude-haiku-4-5-20251001',
            system = self.system,
            max_tokens = 500,
            messages = self.messages,
            stop_sequences=["PAUSE"],
        )
        return response.content[0].text

In [49]:
system = '''
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calc:
e.g. calc(4,7,'*')
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

search:
e.g. search('capital of france')
returns city name which is the capital of france

You are not permitted to answer using your own knowledge even if you are confident. Answer only using the tools provided
Example session:

Question: What is the capital of france?
Thought: I should look capital of france using search("capital of france")
Action: search('capital of france')
PAUSE

You will be called again with this:

Observation: Paris

You then output:

Answer: The capital of france is Paris
'''.strip()

In [50]:
def calc(a,b,op):
    match op:
        case '+':
            return a+b
        case '-':
            return a-b
        case '*':
            return a*b
        case '/':
            return a/b
        case _:
            return 'Unknown operator'

def search(s:str):
    search_data = {
        "capital of france": "Paris",
        "who wrote hamlet": "William Shakespeare",
        "tallest mountain": "Mount Everest",
        "speed of light": "About 299,792,458 meters per second",
        "population of japan": "About 123 million people",
        "largest ocean": "The Pacific Ocean",
        "inventor of the telephone": "Alexander Graham Bell",
        "boiling point of water": "100 degrees Celsius at sea level",
        "smallest planet": "Mercury",
        "author of 1984": "George Orwell",
    }
    if s.lower() in search_data:
        return search_data[s.lower()]
    else:
        return "not found"
    
known_actions = {
    'calc':calc,
    'search':search,
}

In [51]:
model = Agent(system)

In [31]:
result = model('what is the name of the startup that started on 25th august, 2017?')
print(result)

Thought: This question is asking me to identify a startup based on a specific start date of August 25th, 2017. This is too vague - there could be many startups that started on that date. Without more context or specific information about which startup you're referring to, I cannot determine which one you mean. My search function is designed for general knowledge queries, but this would require either knowing which specific startup you're asking about or having access to a comprehensive startup database with specific founding dates, which I don't have.

Answer: I'm unable to determine which startup you're referring to based only on the date August 25th, 2017. There could be many startups that were founded on that date. Could you provide more context or details about the startup you're asking about, such as its industry, location, or any other identifying information?


In [52]:
action_re = re.compile(r'^Action: (\w+)\((.*)\)$')

In [53]:
def query(question, max_turns = 5):
    model = Agent(system)
    next_prompt = question
    for _ in range(max_turns):
        result = model(next_prompt)
        print(result)
        actions = [
            match for a in result.split('\n') if (match := action_re.match(a))
        ]
        if actions:
            action, action_args = actions[0].groups()
            if action not in known_actions:
                raise Exception(f'Unknown action: {action}')
            print(f'running {action} on {action_args}')
            if action == 'calc':
                action_args = action_args.replace(' ','').replace("'",'').split(',')
                observation = known_actions[action](int(action_args[0]),int(action_args[1]),action_args[2])
            else:
                observation = known_actions[action](action_args.replace("'",'').lower())
            print(f'Observation: {observation}')
            next_prompt = f'Observation: {observation}'
        else:
            return


In [55]:
question = "What's the boiling point of water? Add 50 to that number. Also, who is the inventor of the telephone?"
query(question)

Thought: I need to answer three things:
1. The boiling point of water
2. Add 50 to that number
3. Who invented the telephone

However, I'm instructed to only use the tools provided. Let me check what I can do:
- calc: can perform calculations
- search: can search for information

I don't have the boiling point of water stored, and I need to search for it. I also need to search for who invented the telephone. Let me start with these searches.

Action: search('boiling point of water')

running search on 'boiling point of water'
Observation: 100 degrees Celsius at sea level
Thought: Good, I found that the boiling point of water is 100 degrees Celsius. Now I need to:
1. Add 50 to 100
2. Search for who invented the telephone

Let me do the calculation first.

Action: calc(100, 50, '+')


running calc on 100, 50, '+'
Observation: 150
Thought: Great! I got 150 from adding 50 to the boiling point of water (100). Now I need to find out who invented the telephone.

Action: search('inventor of th